In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :exponential

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [exponential_model] Fitting chain 3 (tau=34)
[ Info: [exponential] iter 1000/1000000 elapsed=3.9s, rate=0.174, mean=[1.251, 0.00137, 1.389, 0.406], std=[0.1976, 0.000442, 0.2819, 0.2857] [ADAPT]
[ Info: [exponential] iter 2000/1000000 elapsed=6.9s, rate=0.148, mean=[1.481, 0.00132, 1.548, 0.246], std=[0.2591, 0.000327, 0.2454, 0.2478] [ADAPT]
[ Info: [exponential] iter 3000/1000000 elapsed=9.1s, rate=0.130, mean=[1.649, 0.00124, 1.605, 0.193], std=[0.3060, 0.000297, 0.2136, 0.2132] [ADAPT]
[ Info: [exponential] iter 4000/1000000 elapsed=11.3s, rate=0.118, mean=[1.765, 0.00118, 1.634, 0.166], std=[0.3210, 0.000279, 0.1904, 0.1892] [ADAPT]
[ Info: [exponential] iter 5000/1000000 elapsed=13.4s, rate=0.112, mean=[1.852, 0.00113, 1.651, 0.150], std=[0.3270, 0.000266, 0.1731, 0.1718] [ADAPT]
[ Info: [exponential] iter 6000/1000000 elapsed=15.6s, rate=0.108, mean=[1.895, 0.00111, 1.658, 0.139], std=[0.3107, 0.000251, 0.1587, 0.1582] [ADAPT]
[ Info: [exponential] iter 7000/1000000 elap